In [2]:
# %%
# =========================================================
# FEDERATED LEARNING - FedAvg (Flower) untuk Dataset Drebin
# =========================================================
# Bagian preprocessing (1-10) mengikuti notebook cleaning sebelumnya.
# Bagian ini menambahkan:
#   - Partisi data IID
#   - Partisi data Non-IID (Dirichlet / label-skew)
#   - Model PyTorch sederhana (MLP) untuk klasifikasi biner malware/benign
#   - Flower NumPyClient
#   - Strategy FedAvg + simulasi
 
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import OrderedDict
from typing import Dict, List, Tuple
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score
 
from flwr.client import Client, ClientApp, NumPyClient
from flwr.common import ndarrays_to_parameters, Context, NDArrays, Scalar
from flwr.server import ServerApp, ServerConfig, ServerAppComponents
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation
 
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

2026-09-01 17:25:51,477	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
# =========================================================
# Konfigurasi Device - DIPAKSA CPU
# =========================================================
# Seluruh proses (training lokal maupun simulasi Flower) dijalankan
# di CPU saja, terlepas apakah GPU tersedia di mesin atau tidak.
 
DEVICE = torch.device("cpu")
NUM_GPUS_AVAILABLE = 0  # dipaksa 0 supaya alokasi resource simulasi juga tidak memakai GPU
 
print(f"[INFO] Device yang digunakan: {DEVICE}")
print("[INFO] Menjalankan di CPU (dipaksa, GPU tidak digunakan).")
 
 
# %%
# %% [markdown]
# ## 1. Load & Bersihkan Dataset (ringkas dari notebook sebelumnya)
 
# %%
DATA_PATH = "../dataset/drebin_dataset.csv"
 
data_raw = pd.read_csv(DATA_PATH)
data = data_raw.copy()
 
if "Unnamed: 0" in data.columns:
    data = data.drop(columns=["Unnamed: 0"])
 
SPECIAL_COLUMN = "TelephonyManager.getSimCountryIso"
if SPECIAL_COLUMN in data.columns:
    data[SPECIAL_COLUMN] = data[SPECIAL_COLUMN].replace("?", np.nan)
    data[SPECIAL_COLUMN] = pd.to_numeric(data[SPECIAL_COLUMN], errors="coerce")
 
if "class" not in data.columns:
    raise KeyError(f"Kolom 'class' tidak ditemukan. Kolom tersedia: {list(data.columns)}")
 
data["class"] = data["class"].replace({"B": 0, "S": 1})
 
feature_columns = [c for c in data.columns if c != "class"]
data[feature_columns] = data[feature_columns].apply(pd.to_numeric, errors="coerce")
 
data = data.dropna().reset_index(drop=True)
 
X = data.drop(columns=["class"]).values.astype(np.float32)
y = data["class"].astype(int).values
 
print("Ukuran X:", X.shape, "| Ukuran y:", y.shape)
print("Distribusi kelas:", np.bincount(y))

C:\Users\s2\AppData\Local\Temp\ipykernel_784\3405032801.py:21: DtypeWarning: Columns (0: TelephonyManager.getSimCountryIso) have mixed types. Specify dtype option on import or set low_memory=False.
  data_raw = pd.read_csv(DATA_PATH)


[INFO] Device yang digunakan: cpu
[INFO] Menjalankan di CPU (dipaksa, GPU tidak digunakan).
Ukuran X: (15031, 215) | Ukuran y: (15031,)
Distribusi kelas: [9476 5555]


In [4]:
# %% [markdown]
# ## 2. Train/Test Split Global + Standardisasi
 
# %%
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
 
scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)
 
N_FEATURES = X_train_full.shape[1]
N_CLASSES = 2

In [5]:
# %%
# %% [markdown]
# ## 3. Fungsi Partisi Data: IID dan Non-IID
 
# %%
def partition_iid(
    X: np.ndarray, y: np.ndarray, num_clients: int, seed: int = SEED
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Bagi data secara acak merata (IID) ke semua client."""
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y))
    rng.shuffle(idx)
    splits = np.array_split(idx, num_clients)
    return [(X[s], y[s]) for s in splits]
 
 
def partition_non_iid_dirichlet(
    X: np.ndarray,
    y: np.ndarray,
    num_clients: int,
    alpha: float = 0.3,
    seed: int = SEED,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """
    Bagi data secara non-IID menggunakan distribusi Dirichlet per kelas.
    alpha kecil (misal 0.1-0.5) -> distribusi label makin timpang antar client.
    alpha besar (misal 10+) -> mendekati IID.
    """
    rng = np.random.default_rng(seed)
    n_classes = len(np.unique(y))
    client_indices = [[] for _ in range(num_clients)]
 
    for c in range(n_classes):
        idx_c = np.where(y == c)[0]
        rng.shuffle(idx_c)
 
        # proporsi tiap client untuk kelas c, ditarik dari Dirichlet
        proportions = rng.dirichlet(alpha=np.repeat(alpha, num_clients))
        # ubah proporsi jadi titik potong (cumulative)
        cut_points = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        splits = np.split(idx_c, cut_points)
 
        for client_id, split in enumerate(splits):
            client_indices[client_id].extend(split.tolist())
 
    partitions = []
    for indices in client_indices:
        indices = np.array(indices, dtype=int)
        rng.shuffle(indices)
        partitions.append((X[indices], y[indices]))
 
    return partitions
 
 
def print_partition_summary(partitions: List[Tuple[np.ndarray, np.ndarray]], title: str):
    print(f"\n================ {title} ================")
    for i, (Xi, yi) in enumerate(partitions):
        counts = np.bincount(yi, minlength=N_CLASSES)
        print(f"Client {i}: total={len(yi)} | kelas 0={counts[0]} | kelas 1={counts[1]}")
 

In [6]:
 
# %%
# %% [markdown]
# ## 4. Melihat Hasil Partisi IID dan Non-IID
 
NUM_CLIENTS = 5
ALPHA = 0.3
 
# Partisi IID
iid_partitions = partition_iid(
    X_train_full,
    y_train_full,
    num_clients=NUM_CLIENTS,
    seed=SEED
)
 
# Partisi Non-IID
non_iid_partitions = partition_non_iid_dirichlet(
    X_train_full,
    y_train_full,
    num_clients=NUM_CLIENTS,
    alpha=ALPHA,
    seed=SEED
)
 
 
# %% [markdown]
# ### Fungsi untuk menampilkan statistik partisi
 
# %%
def show_partition_stats(partitions, name="Partition"):
    print("=" * 70)
    print(f" {name}")
    print("=" * 70)
 
    total_data = sum(len(y_client) for _, y_client in partitions)
 
    for client_id, (X_client, y_client) in enumerate(partitions):
        unique, counts = np.unique(y_client, return_counts=True)
        label_counts = dict(zip(unique, counts))
 
        count_0 = label_counts.get(0, 0)
        count_1 = label_counts.get(1, 0)
 
        pct_0 = count_0 / len(y_client) * 100
        pct_1 = count_1 / len(y_client) * 100
 
        print(
            f"Client {client_id + 1}: "
            f"Total={len(y_client):5d} | "
            f"Label 0={count_0:5d} ({pct_0:6.2f}%) | "
            f"Label 1={count_1:5d} ({pct_1:6.2f}%)"
        )
 
    print("-" * 70)
    print(f"Total data : {total_data}")
    print("=" * 70)
 
 
# %% [markdown]
# ### Output IID
 
# %%
show_partition_stats(
    iid_partitions,
    name="IID Partition"
)
 
 
# %% [markdown]
# ### Output Non-IID
 
# %%
show_partition_stats(
    non_iid_partitions,
    name=f"Non-IID Dirichlet (alpha={ALPHA})"
)
 

 IID Partition
Client 1: Total= 2405 | Label 0= 1525 ( 63.41%) | Label 1=  880 ( 36.59%)
Client 2: Total= 2405 | Label 0= 1501 ( 62.41%) | Label 1=  904 ( 37.59%)
Client 3: Total= 2405 | Label 0= 1523 ( 63.33%) | Label 1=  882 ( 36.67%)
Client 4: Total= 2405 | Label 0= 1477 ( 61.41%) | Label 1=  928 ( 38.59%)
Client 5: Total= 2404 | Label 0= 1554 ( 64.64%) | Label 1=  850 ( 35.36%)
----------------------------------------------------------------------
Total data : 12024
 Non-IID Dirichlet (alpha=0.3)
Client 1: Total= 5639 | Label 0= 4250 ( 75.37%) | Label 1= 1389 ( 24.63%)
Client 2: Total= 1364 | Label 0=  629 ( 46.11%) | Label 1=  735 ( 53.89%)
Client 3: Total= 2159 | Label 0= 2111 ( 97.78%) | Label 1=   48 (  2.22%)
Client 4: Total=  533 | Label 0=  482 ( 90.43%) | Label 1=   51 (  9.57%)
Client 5: Total= 2329 | Label 0=  108 (  4.64%) | Label 1= 2221 ( 95.36%)
----------------------------------------------------------------------
Total data : 12024


In [7]:

# %% [markdown]
# ## 4. Konfigurasi Jumlah Client & Pilih Mode Partisi
 
# %%
NUM_CLIENTS = 5
NUM_ROUNDS = 10
LOCAL_EPOCHS = 50
BATCH_SIZE = 32
 
# Ganti PARTITION_MODE ke "iid" atau "non_iid" sesuai eksperimen yang diinginkan
PARTITION_MODE = "non_iid"  # "iid" | "non_iid"
DIRICHLET_ALPHA = 0.3  # dipakai jika PARTITION_MODE == "non_iid"
 
if PARTITION_MODE == "iid":
    client_partitions = partition_iid(X_train_full, y_train_full, NUM_CLIENTS)
    print_partition_summary(client_partitions, "PARTISI IID")
else:
    client_partitions = partition_non_iid_dirichlet(
        X_train_full, y_train_full, NUM_CLIENTS, alpha=DIRICHLET_ALPHA
    )
    print_partition_summary(client_partitions, "PARTISI NON-IID (Dirichlet)")
 


================ PARTISI NON-IID (Dirichlet) ================
Client 0: total=5639 | kelas 0=4250 | kelas 1=1389
Client 1: total=1364 | kelas 0=629 | kelas 1=735
Client 2: total=2159 | kelas 0=2111 | kelas 1=48
Client 3: total=533 | kelas 0=482 | kelas 1=51
Client 4: total=2329 | kelas 0=108 | kelas 1=2221


In [8]:

# %% [markdown]
# ## Federated Learning
 
# %%
# %% [markdown]
# ## 5. Model PyTorch (MLP sederhana untuk klasifikasi biner)
 
# %%
class DrebinMLP(nn.Module):
    def __init__(self, n_features: int, n_classes: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )
 
    def forward(self, x):
        return self.net(x)
 
 
def get_parameters(model: nn.Module) -> NDArrays:
    return [val.cpu().numpy() for _, val in model.state_dict().items()]
 
 
def set_parameters(model: nn.Module, parameters: NDArrays) -> None:
    params_dict = zip(model.state_dict().keys(), parameters)
    state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
    model.load_state_dict(state_dict, strict=True)
 
 
def train(model: nn.Module, X: np.ndarray, y: np.ndarray, epochs: int, batch_size: int, device: torch.device = DEVICE):
    model.to(device)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
 
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    dataset = torch.utils.data.TensorDataset(X_t, y_t)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
 
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
 
 
def test(model: nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 64, device: torch.device = DEVICE) -> Tuple[float, Dict[str, float]]:
    """
    Evaluasi model dan hitung loss + accuracy + precision + recall + f1-score.
    Precision/recall/f1 dihitung dengan average="binary" (pos_label=1 -> kelas malware/S).
    Jika hanya ada 1 kelas pada batch data (edge case), fallback ke average="macro" dengan zero_division=0.
    """
    model.to(device)
    model.eval()
    criterion = nn.CrossEntropyLoss()
 
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    dataset = torch.utils.data.TensorDataset(X_t, y_t)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)
 
    total_loss, correct, total = 0.0, 0, 0
    all_preds: List[int] = []
    all_labels: List[int] = []
 
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * len(yb)
            preds = torch.argmax(out, dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(yb.cpu().numpy().tolist())
 
    avg_loss = total_loss / total if total > 0 else 0.0
    accuracy = correct / total if total > 0 else 0.0
 
    n_unique = len(set(all_labels))
    avg_mode = "binary" if n_unique > 1 else "macro"
 
    precision = precision_score(
        all_labels, all_preds, average=avg_mode, pos_label=1, zero_division=0
    )
    recall = recall_score(
        all_labels, all_preds, average=avg_mode, pos_label=1, zero_division=0
    )
    f1 = f1_score(
        all_labels, all_preds, average=avg_mode, pos_label=1, zero_division=0
    )
 
    metrics = {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
    }
    return avg_loss, metrics
 
# %%
# %% [markdown]
# ### Melihat Struktur Model
 
# %%
model = DrebinMLP(
    n_features=N_FEATURES,
    n_classes=N_CLASSES
)
 
print(model)
 
# %%
total_params = sum(
    p.numel() for p in model.parameters()
)
 
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)
 
print(f"Total parameter     : {total_params:,}")
print(f"Trainable parameter : {trainable_params:,}")
 
# %% [markdown]
# ### Melihat Parameter Model
 
# %%
parameters = get_parameters(model)
 
print("Jumlah parameter tensors:", len(parameters))
 
for i, param in enumerate(parameters):
    print(
        f"Parameter {i}: "
        f"shape={param.shape}, "
        f"dtype={param.dtype}"
    )
 
 
# %%
# %%
for i, param in enumerate(parameters):
    print(f"Parameter {i}:")
    print(param.flatten()[:10])
 
 
# %%
# %%
print("Isi state_dict model:")
 
for name, param in model.state_dict().items():
    print(
        f"{name:20s} "
        f"shape={tuple(param.shape)} "
        f"dtype={param.dtype}"
    )
 
 

DrebinMLP(
  (net): Sequential(
    (0): Linear(in_features=215, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=2, bias=True)
  )
)
Total parameter     : 36,034
Trainable parameter : 36,034
Jumlah parameter tensors: 6
Parameter 0: shape=(128, 215), dtype=float32
Parameter 1: shape=(128,), dtype=float32
Parameter 2: shape=(64, 128), dtype=float32
Parameter 3: shape=(64,), dtype=float32
Parameter 4: shape=(2, 64), dtype=float32
Parameter 5: shape=(2,), dtype=float32
Parameter 0:
[ 0.05214109  0.05660607 -0.01597725  0.06264877 -0.01494274  0.01376201
 -0.03320324  0.04005235  0.06012072 -0.05003303]
Parameter 1:
[-0.00049017 -0.03095568 -0.02582384 -0.06124129 -0.02801841 -0.03406459
 -0.02765906 -0.042805   -0.01478758 -0.03704804]
Parameter 2:
[-0.0280253  -0.06952967  0.0592132   0.01623228  0.01018937  0.00084894
  0.08388215  

In [9]:

# %%
# %% [markdown]
# ## 6. Testing Model dan Melihat Output
 
# %%
print("=" * 70)
print("MODEL INFORMATION")
print("=" * 70)
 
model = DrebinMLP(
    n_features=N_FEATURES,
    n_classes=N_CLASSES
)
 
print(model)
 
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
 
 
# %% [markdown]
# ### Parameter Model
 
# %%
print("\n" + "=" * 70)
print("MODEL PARAMETERS")
print("=" * 70)
 
parameters = get_parameters(model)
 
for i, param in enumerate(parameters):
    print(
        f"Parameter {i}: "
        f"shape={param.shape}, "
        f"min={param.min():.6f}, "
        f"max={param.max():.6f}, "
        f"mean={param.mean():.6f}"
    )
 
 
# %% [markdown]
# ### Forward Pass
 
# %%
print("\n" + "=" * 70)
print("FORWARD PASS")
print("=" * 70)
 
X_sample = X_train_full[:5]
y_sample = y_train_full[:5]
 
X_tensor = torch.tensor(
    X_sample,
    dtype=torch.float32
)
 
model.eval()
 
with torch.no_grad():
    logits = model(X_tensor)
    probabilities = torch.softmax(logits, dim=1)
    predictions = torch.argmax(logits, dim=1)
 
print("Input shape       :", X_tensor.shape)
print("Logits shape      :", logits.shape)
 
print("\nLogits:")
print(logits)
 
print("\nProbabilities:")
print(probabilities)
 
print("\nPredictions :", predictions.numpy())
print("True labels :", y_sample)

MODEL INFORMATION
DrebinMLP(
  (net): Sequential(
    (0): Linear(in_features=215, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=2, bias=True)
  )
)

Total parameters: 36,034

MODEL PARAMETERS
Parameter 0: shape=(128, 215), min=-0.068196, max=0.068190, mean=0.000263
Parameter 1: shape=(128,), min=-0.067979, max=0.066763, mean=-0.004542
Parameter 2: shape=(64, 128), min=-0.088337, max=0.088369, mean=0.000680
Parameter 3: shape=(64,), min=-0.087001, max=0.082073, mean=-0.014593
Parameter 4: shape=(2, 64), min=-0.124831, max=0.124114, mean=-0.009570
Parameter 5: shape=(2,), min=0.076367, max=0.092114, mean=0.084240

FORWARD PASS
Input shape       : torch.Size([5, 215])
Logits shape      : torch.Size([5, 2])

Logits:
tensor([[0.0667, 0.0636],
        [0.0556, 0.0497],
        [0.0016, 0.0222],
        [0.0851, 0.0198],
        [0.11

In [10]:

# %%
print("\n" + "=" * 70)
print("TRAINING")
print("=" * 70)
 
train(
    model,
    X_train_full,
    y_train_full,
    epochs=5,
    batch_size=64
)
 
print("Training selesai!")
 
 
# %% [markdown]
# ### Testing
 
# %%
print("\n" + "=" * 70)
print("TESTING")
print("=" * 70)
 
test_loss, test_metrics = test(
    model,
    X_test,
    y_test
)
 
print(f"Test Loss : {test_loss:.4f}")
 
for name, value in test_metrics.items():
    print(f"{name:10s}: {value:.4f}")
 
 
# %%
# =========================================================
# Perhitungan Bandwidth (Communication Cost)
# =========================================================
def compute_parameters_size_bytes(parameters: NDArrays) -> int:
    """Hitung total ukuran (byte) dari sekumpulan array parameter model."""
    return int(sum(arr.nbytes for arr in parameters))
 
 
def bytes_to_mb(num_bytes: int) -> float:
    return num_bytes / (1024 ** 2)
 
 
# %% [markdown]
# ## 6. Flower Client
 
# %%
class DrebinClient(NumPyClient):
    def __init__(self, X_train, y_train, X_val, y_val, device: torch.device = DEVICE):
        self.model = DrebinMLP(N_FEATURES, N_CLASSES)
        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.device = device
 
    def get_parameters(self, config: Dict[str, Scalar]) -> NDArrays:
        return get_parameters(self.model)
 
    def fit(self, parameters: NDArrays, config: Dict[str, Scalar]):
        set_parameters(self.model, parameters)
        train(
            self.model, self.X_train, self.y_train,
            epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, device=self.device,
        )
        return get_parameters(self.model), len(self.X_train), {}
 
    def evaluate(self, parameters: NDArrays, config: Dict[str, Scalar]):
        set_parameters(self.model, parameters)
        loss, metrics = test(self.model, self.X_val, self.y_val, device=self.device)
        return float(loss), len(self.X_val), metrics
 
 
def client_fn(context: Context) -> Client:
    partition_id = int(context.node_config["partition-id"])
    X_client, y_client = client_partitions[partition_id]
 
    # split lokal train/val per client (80/20)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_client, y_client, test_size=0.2, random_state=SEED,
        stratify=y_client if len(np.unique(y_client)) > 1 else None,
    )
    return DrebinClient(X_tr, y_tr, X_val, y_val, device=DEVICE).to_client()
 
 
client_app = ClientApp(client_fn=client_fn)
 
# %%
# %% [markdown]
# ## 7. Evaluasi Global (di server, pakai X_test/y_test)
 
# %%
def get_evaluate_fn(X_test: np.ndarray, y_test: np.ndarray):
    def evaluate(server_round: int, parameters: NDArrays, config: Dict[str, Scalar]):
        model = DrebinMLP(N_FEATURES, N_CLASSES)
        set_parameters(model, parameters)
        loss, metrics = test(model, X_test, y_test, device=DEVICE)
        print(
            f"[Round {server_round}] Global Test | "
            f"loss={loss:.4f} | acc={metrics['accuracy']:.4f} | "
            f"precision={metrics['precision']:.4f} | recall={metrics['recall']:.4f} | "
            f"f1={metrics['f1_score']:.4f}"
        )
        return loss, metrics
 
    return evaluate
 
 
def weighted_average(metrics: List[Tuple[int, Dict[str, Scalar]]]) -> Dict[str, Scalar]:
    """Agregasi weighted-average untuk accuracy, precision, recall, f1 dari semua client."""
    examples = [num_examples for num_examples, _ in metrics]
    total_examples = sum(examples)
 
    aggregated: Dict[str, Scalar] = {}
    for key in ["accuracy", "precision", "recall", "f1_score"]:
        weighted_sum = sum(num_examples * m[key] for num_examples, m in metrics if key in m)
        aggregated[key] = weighted_sum / total_examples if total_examples > 0 else 0.0
 
    return aggregated


TRAINING
Training selesai!

TESTING
Test Loss : 0.0358
accuracy  : 0.9877
precision : 0.9909
recall    : 0.9757
f1_score  : 0.9832


In [11]:

# %% [markdown]
# ## 6. Flower Client
 
# %%
class DrebinClient(NumPyClient):
    def __init__(self, X_train, y_train, X_val, y_val, device: torch.device = DEVICE):
        self.model = DrebinMLP(N_FEATURES, N_CLASSES)
        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.device = device
 
    def get_parameters(self, config: Dict[str, Scalar]) -> NDArrays:
        return get_parameters(self.model)
 
    def fit(self, parameters: NDArrays, config: Dict[str, Scalar]):
        set_parameters(self.model, parameters)
        train(
            self.model, self.X_train, self.y_train,
            epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, device=self.device,
        )
        return get_parameters(self.model), len(self.X_train), {}
 
    def evaluate(self, parameters: NDArrays, config: Dict[str, Scalar]):
        set_parameters(self.model, parameters)
        loss, metrics = test(self.model, self.X_val, self.y_val, device=self.device)
        return float(loss), len(self.X_val), metrics
 
 
def client_fn(context: Context) -> Client:
    partition_id = int(context.node_config["partition-id"])
    X_client, y_client = client_partitions[partition_id]
 
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_client, y_client, test_size=0.2, random_state=SEED,
        stratify=y_client if len(np.unique(y_client)) > 1 else None,
    )
    return DrebinClient(X_tr, y_tr, X_val, y_val, device=DEVICE).to_client()
 
 
client_app = ClientApp(client_fn=client_fn)
 
# %%
# %% [markdown]
# ## 7. Evaluasi Global (di server, pakai X_test/y_test)
 
# %%
def get_evaluate_fn(X_test: np.ndarray, y_test: np.ndarray):
    def evaluate(server_round: int, parameters: NDArrays, config: Dict[str, Scalar]):
        model = DrebinMLP(N_FEATURES, N_CLASSES)
        set_parameters(model, parameters)
        loss, metrics = test(model, X_test, y_test, device=DEVICE)
        print(
            f"[Round {server_round}] Global Test | "
            f"loss={loss:.4f} | acc={metrics['accuracy']:.4f} | "
            f"precision={metrics['precision']:.4f} | recall={metrics['recall']:.4f} | "
            f"f1={metrics['f1_score']:.4f}"
        )
        return loss, metrics
 
    return evaluate
 
 
def weighted_average(metrics: List[Tuple[int, Dict[str, Scalar]]]) -> Dict[str, Scalar]:
    """Agregasi weighted-average untuk accuracy, precision, recall, f1 dari semua client."""
    examples = [num_examples for num_examples, _ in metrics]
    total_examples = sum(examples)
 
    aggregated: Dict[str, Scalar] = {}
    for key in ["accuracy", "precision", "recall", "f1_score"]:
        weighted_sum = sum(num_examples * m[key] for num_examples, m in metrics if key in m)
        aggregated[key] = weighted_sum / total_examples if total_examples > 0 else 0.0
 
    return aggregated
 
 
# %% [markdown]
# ## 8. Server App dengan Strategy FedAvg BAWAAN (tanpa custom class)
 
# %%
def server_fn(context: Context) -> ServerAppComponents:
    initial_model = DrebinMLP(N_FEATURES, N_CLASSES)
    initial_parameters = ndarrays_to_parameters(get_parameters(initial_model))
 
    strategy = FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
        evaluate_fn=get_evaluate_fn(X_test, y_test),
        evaluate_metrics_aggregation_fn=weighted_average,
        fit_metrics_aggregation_fn=weighted_average,
    )
 
    config = ServerConfig(num_rounds=NUM_ROUNDS)
    return ServerAppComponents(strategy=strategy, config=config)
 
 
server_app = ServerApp(server_fn=server_fn)
 
# %% [markdown]
# ## 9. Jalankan Simulasi
 
# %%
if __name__ == "__main__":
    backend_config = {"client_resources": {"num_cpus": 1, "num_gpus": 0.0}}
    print("[INFO] Simulasi memakai CPU saja untuk semua client.")
 
    run_simulation(
        server_app=server_app,
        client_app=client_app,
        num_supernodes=NUM_CLIENTS,
        backend_config=backend_config,
    )
 
    # =========================================================
    # 10. Estimasi Bandwidth Komunikasi (dihitung terpisah, sederhana)
    # =========================================================
    # Bandwidth komunikasi FL pada FedAvg standar:
    #   - Setiap round, server mengirim parameter model global ke semua
    #     client yang berpartisipasi (DOWNLOAD).
    #   - Setiap client mengirim balik parameter hasil training lokal
    #     ke server (UPLOAD).
    # Karena strategy yang dipakai adalah FedAvg bawaan (tidak mencatat
    # bandwidth per round secara live), estimasi ini dihitung setelah
    # training selesai berdasarkan: ukuran 1 salinan parameter model x
    # jumlah client x jumlah round x 2 arah (download + upload).
 
    print("\n================ ESTIMASI BANDWIDTH KOMUNIKASI ================")
 
    final_model = DrebinMLP(N_FEATURES, N_CLASSES)
    param_size_bytes = sum(arr.nbytes for arr in get_parameters(final_model))
    param_size_mb = param_size_bytes / (1024 ** 2)
 
    download_mb_per_round = param_size_mb * NUM_CLIENTS
    upload_mb_per_round = param_size_mb * NUM_CLIENTS
    total_mb_per_round = download_mb_per_round + upload_mb_per_round
 
    total_download_mb = download_mb_per_round * NUM_ROUNDS
    total_upload_mb = upload_mb_per_round * NUM_ROUNDS
    total_bandwidth_mb = total_mb_per_round * NUM_ROUNDS
 
    print(f"Ukuran 1 salinan parameter model : {param_size_mb:.3f} MB")
    print(f"Jumlah client per round          : {NUM_CLIENTS}")
    print(f"Jumlah round                     : {NUM_ROUNDS}")
    print("-" * 66)
    print(f"Download per round (server->client) : {download_mb_per_round:.3f} MB")
    print(f"Upload per round   (client->server) : {upload_mb_per_round:.3f} MB")
    print(f"Total per round                     : {total_mb_per_round:.3f} MB")
    print("-" * 66)
    print(f"Total DOWNLOAD selama {NUM_ROUNDS} round : {total_download_mb:.3f} MB")
    print(f"Total UPLOAD   selama {NUM_ROUNDS} round : {total_upload_mb:.3f} MB")
    print(f"Total BANDWIDTH keseluruhan               : {total_bandwidth_mb:.3f} MB")

[INFO] Simulasi memakai CPU saja untuk semua client.


INFO :      Starting Flower ServerApp, config: num_rounds=10, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Evaluating initial global parameters
INFO :      initial parameters (loss, other metrics): 0.6566605917789787, {'accuracy': 0.627203192550715, 'precision': 0.25, 'recall': 0.004500450045004501, 'f1_score': 0.008841732979664015}
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 5 clients (out of 5)


[Round 0] Global Test | loss=0.6567 | acc=0.6272 | precision=0.2500 | recall=0.0045 | f1=0.0088


(raylet) Windows fatal exception: Windows fatal exception: access violation
(raylet) 
(raylet) Stack (most recent call first):
(raylet)   File "Windows fatal exception: access violation
(raylet) 
(raylet) c:\Users\s2\Documents\FedMal\.fed\Lib\site-packages\ray\_private\worker.py", line 627 in job_logging_config
(raylet)   File "c:\Users\s2\Documents\FedMal\.fed\Lib\site-packages\ray\_private\worker.py", line 2807 in disconnect
(raylet)   File "c:\Users\s2\Documents\FedMal\.fed\Lib\site-packages\ray\_private\worker.py", line 2118 in shutdown
(raylet)   File "c:\Users\s2\Documents\FedMal\.fed\Lib\site-packages\ray\_private\worker.py", line 1170 in Windows fatal exception: access violation
(raylet) 
(raylet) wrapper
(raylet)   File "Stack (most recent call first):
(raylet)   File "c:\Users\s2\Documents\FedMal\.fed\Lib\site-packages\ray\_private\worker.py", line 627 in job_logging_config
(raylet)   File "c:\Users\s2\Documents\FedMal\.fed\Lib\site-packages\ray\_private\worker.py", line 2807

[Round 1] Global Test | loss=0.0651 | acc=0.9834 | precision=0.9836 | recall=0.9712 | f1=0.9774


...

[Round 9] Global Test | loss=0.1350 | acc=0.9870 | precision=0.9846 | recall=0.9802 | f1=0.9824


INFO :      aggregate_evaluate: received 5 results and 0 failures
INFO :      
INFO :      [ROUND 10]
INFO :      configure_fit: strategy sampled 5 clients (out of 5)
INFO :      aggregate_fit: received 5 results and 0 failures
INFO :      fit progress: (10, 0.15507430906239822, {'accuracy': 0.9876953774526106, 'precision': 0.9846570397111913, 'recall': 0.981998199819982, 'f1_score': 0.9833258224425416}, 115.59337719999894)
INFO :      configure_evaluate: strategy sampled 5 clients (out of 5)


[Round 10] Global Test | loss=0.1551 | acc=0.9877 | precision=0.9847 | recall=0.9820 | f1=0.9833


INFO :      aggregate_evaluate: received 5 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 10 round(s) in 116.60s
INFO :      	History (loss, distributed):
INFO :      		round 1: 0.10891479256168837
INFO :      		round 2: 0.14013225728414058
INFO :      		round 3: 0.16398453727911633
INFO :      		round 4: 0.17035712570175843
INFO :      		round 5: 0.19484468346261533
INFO :      		round 6: 0.21990149609817017
INFO :      		round 7: 0.1953401057614677
INFO :      		round 8: 0.21101252218039393
INFO :      		round 9: 0.21542392880507946
INFO :      		round 10: 0.22481599557437473
INFO :      	History (loss, centralized):
INFO :      		round 0: 0.6566605917789787
INFO :      		round 1: 0.06505868298233566
INFO :      		round 2: 0.090786424292345
INFO :      		round 3: 0.10939756977905515
INFO :      		round 4: 0.10690921747522435
INFO :      		round 5: 0.1329297061785337
INFO :      		round 6: 0.14808087929105185
INFO :      		round 7: 0.140402882562167


================ ESTIMASI BANDWIDTH KOMUNIKASI ================
Ukuran 1 salinan parameter model : 0.137 MB
Jumlah client per round          : 5
Jumlah round                     : 10
------------------------------------------------------------------
Download per round (server->client) : 0.687 MB
Upload per round   (client->server) : 0.687 MB
Total per round                     : 1.375 MB
------------------------------------------------------------------
Total DOWNLOAD selama 10 round : 6.873 MB
Total UPLOAD   selama 10 round : 6.873 MB
Total BANDWIDTH keseluruhan               : 13.746 MB
